# Merge NSRDB station files from Google Drive

This notebook reads the station list from CYL_geo.csv, finds matching NSRDB CSV files for each station code, skips the first two rows in each source file, uses the third row as the header, combines them, and saves one merged CSV per station in the merged_files2 folder.

In [ ]:
import os
import glob
import pandas as pd

# Mount Google Drive when running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    base_dir = '/content/drive/MyDrive'
except Exception:
    base_dir = os.path.expanduser('~/MyDrive')

geo_path = os.path.join(base_dir, 'CYL_geo', 'CyL_geo.csv')
nsrdb_dir = os.path.join(base_dir, 'NSRDB')
merged_dir = os.path.join(nsrdb_dir, 'merged_files2')

if not os.path.exists(geo_path):
    raise FileNotFoundError(f'Geo file not found: {geo_path}')

os.makedirs(merged_dir, exist_ok=True)

geo_df = pd.read_csv(geo_path)
if 'station_code' not in geo_df.columns:
    raise KeyError('The file must contain a station_code column.')

station_codes = [str(code).strip() for code in geo_df['station_code'].dropna().unique() if str(code).strip()]

print(f'Found {len(station_codes)} station codes in {geo_path}')

for station_code in station_codes:
    pattern = os.path.join(nsrdb_dir, '**', f'{station_code}_*.csv')
    files = sorted(glob.glob(pattern, recursive=True))

    if not files:
        print(f'No files found for {station_code}')
        continue

    frames = []
    for file_path in files:
        df = pd.read_csv(file_path, skiprows=2)
        frames.append(df)

    merged_df = pd.concat(frames, ignore_index=True, axis=0)

    output_path = os.path.join(merged_dir, f'{station_code}_merged.csv')
    merged_df.to_csv(output_path, index=False)

    print(f'Saved {len(files)} files for {station_code} -> {output_path}')